In [0]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, avg, max as spark_max, min as spark_min, sum as spark_sum,
    to_date, weekofyear, month, year,
    desc, asc, last, row_number
)
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("ReadFromBlob").getOrCreate()

storage_account_name = "stockmarket001"
container_name = "stock-data"


folder_path = "silver/2026/03/22/data.parquet"

input_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{folder_path}"

In [0]:


spark.conf.set(f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net", access_key)

df = spark.read.parquet(input_path)
df.show()

In [0]:

def aggregate_stock_data_with_columns(df: DataFrame,
                                      group_by: str = "daily",
                                      metrics: list = None,
                                      date_col: str = "date",
                                      stock_col: str = "sector") -> DataFrame:
    df = df.withColumn(date_col, to_date(col(date_col), "yyyy-MM-dd"))

    # If no metrics provided, take all numeric columns except date/stock
    if metrics is None:
        excluded_cols = [date_col, stock_col]
        metrics = [c for c, t in df.dtypes if t in ["double", "int", "bigint", "long"] and c not in excluded_cols]

    partition_cols = []
    if stock_col:
        partition_cols.append(stock_col)

    if group_by == "weekly":
        df = df.withColumn("year", year(col(date_col))).withColumn("week", weekofyear(col(date_col)))
        partition_cols += ["year", "week"]
    elif group_by == "monthly":
        df = df.withColumn("year", year(col(date_col))).withColumn("month", month(col(date_col)))
        partition_cols += ["year", "month"]
    else:  # daily
        partition_cols.append(date_col)

    window_spec = Window.partitionBy(*partition_cols)

    for metric in metrics:
        df = df.withColumn(f"{metric}_avg", avg(col(metric)).over(window_spec)) \
               .withColumn(f"{metric}_max", spark_max(col(metric)).over(window_spec)) \
               .withColumn(f"{metric}_min", spark_min(col(metric)).over(window_spec)) \
               .withColumn(f"{metric}_sum", spark_sum(col(metric)).over(window_spec))

    return df

In [0]:
metrics_to_agg = [
    "open", "high", "low", "close",
    "daily_return", "daily_return_pct",
    "volatility_7", "volatility_14", "volatility_30"
]


daily_df = aggregate_stock_data_with_columns(df, group_by="daily", metrics=metrics_to_agg)

weekly_df = aggregate_stock_data_with_columns(df, group_by="weekly", metrics=metrics_to_agg)

monthly_df = aggregate_stock_data_with_columns(df, group_by="monthly", metrics=metrics_to_agg)


In [0]:

def get_top_gainers_losers(df: DataFrame, price_col: str = "daily_return_pct", stock_col: str = "sector", top_n: int = 5):
    

    top_gainers = df.orderBy(desc(price_col)).limit(top_n)

    top_losers = df.orderBy(asc(price_col)).limit(top_n)

    return top_gainers, top_losers

In [0]:
top_gainers, top_losers = get_top_gainers_losers(df, price_col="daily_return_pct", stock_col="sector", top_n=5)

print("Top Gainers:")
top_gainers.show()

print("Top Losers:")
top_losers.show()

In [0]:

def save_as_delta_table(df: DataFrame,
                        table_name: str,
                        mode: str = "overwrite"):

    df.write.format("delta").mode(mode).saveAsTable(table_name)
    print(f" Managed Delta table '{table_name}' created")

In [0]:
save_as_delta_table(daily_df, table_name="gold_daily_stock_data")

save_as_delta_table(weekly_df, table_name="gold_weekly_stock_data")

save_as_delta_table(monthly_df, table_name="gold_monthly_stock_data")

save_as_delta_table(top_gainers, table_name="gold_top_gainers_stock_data")

save_as_delta_table(top_losers, table_name="gold_top_losers_stock_data")

In [0]:
%sql
SELECT date, sector, close
FROM gold_daily_stock_data
WHERE sector = 'Technology'
ORDER BY date

Databricks visualization. Run in Databricks to view.

In [0]:

windowSpec = Window.partitionBy("sector").orderBy("date").rowsBetween(-6, 0)  # 7-day MA

daily_ma = spark.table("gold_daily_stock_data") \
    .withColumn("ma_7", avg("close").over(windowSpec)) \
    .withColumn("ma_14", avg("close").over(Window.partitionBy("sector").orderBy("date").rowsBetween(-13, 0))) \
    .withColumn("ma_30", avg("close").over(Window.partitionBy("sector").orderBy("date").rowsBetween(-29, 0)))

daily_ma.createOrReplaceTempView("daily_ma_view")

In [0]:
%sql
-- Stock price + moving averages for a sector
SELECT date, close, ma_7, ma_14, ma_30
FROM daily_ma_view
WHERE sector = 'Technology'
ORDER BY date;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT date, sector, daily_return_pct
FROM gold_top_gainers_stock_data
ORDER BY daily_return_pct DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Top 10 daily losers
SELECT date, sector, daily_return_pct
FROM gold_top_losers_stock_data
ORDER BY daily_return_pct ASC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Average daily return per sector
SELECT sector, AVG(daily_return_pct) AS avg_return, AVG(volatility_7) AS avg_volatility
FROM gold_daily_stock_data
GROUP BY sector
ORDER BY avg_return DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT date, sector, close_sum AS volume
FROM gold_daily_stock_data
ORDER BY date;

Databricks visualization. Run in Databricks to view.